In [1]:
from dotenv import load_dotenv
import os
from openai import AzureOpenAI  

import importlib
import requests
from minsearch import Index
import json
import pandas as pd

from starter import rag
from rag_helper import RAGBase, llm_client

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor



In [ ]:
provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [8]:
with tracer.start_as_current_span("my_operation") as span:
    query = "How does the agentic loop keep calling the model until it stops?"
    answer = rag.rag(query)
    print(answer)
    span.set_attribute("my_key", "my_value")

('It keeps calling the model inside a `while True` loop.\n\nAfter each model response, the code checks whether the response contains any `function_call` items:\n\n- If yes, it runs the tool, appends the tool output to `messages`, and loops again.\n- If no, it breaks out of the loop.\n\nSo the stop condition is simply: **no function calls in the latest response**.', ResponseUsage(input_tokens=7121, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=117, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7238))
{
    "name": "my_operation",
    "context": {
        "trace_id": "0x4d80badcec70ab6e562f0c0b985db759",
        "span_id": "0x407a025252afe432",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-20T13:22:29.346531Z",
    "end_time": "2026-07-20T13:22:31.905542Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "my_key": "my_value"


In [3]:
# with class
class RAGTraced(RAGBase):
    def rag(self, question: str):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("question", question)
            search_results = self.search(question)
            prompt = self.build_prompt(question, search_results)
            llm_response = self.llm(prompt)
            answer = llm_response.output_text.strip()
            usage = llm_response.usage
            span.set_attribute("answer_length", len(answer))
            return answer, usage
    
    def search(self, query: str, num_results: int = 5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            results = super().search(query, num_results)
            span.set_attribute("num_results_found", len(results))
            return results
    
    def llm(self, prompt: str):
        with tracer.start_as_current_span("llm") as span:
            span.set_attribute("prompt_length", len(prompt))
            response = super().llm(prompt)
            
            # Add token information from the usage object
            if hasattr(response, 'usage') and response.usage:
                span.set_attribute("input_tokens", response.usage.input_tokens)
                span.set_attribute("output_tokens", response.usage.output_tokens)
                
            span.set_attribute("response_length", len(response.output_text))
            return response

In [4]:
load_dotenv()

True

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [6]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [7]:
len(files)

72

In [8]:
# Index the documents with minsearch - make content a text field and filename a keyword field
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index.fit(documents)

In [31]:
# Create an instance of the class and run the query
rag_traced = RAGTraced(index=index,
                       llm_client=llm_client) 
result = rag_traced.rag("How does the agentic loop keep calling the model until it stops?")
answer, usage = result
print("Answer:", answer)
print("Usage:", usage)

{
    "name": "search",
    "context": {
        "trace_id": "0xb6af64ad0ce5b7ccbec1002b8cbd82d1",
        "span_id": "0xe453345bcd7ad213",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2aa606e9018e7654",
    "start_time": "2026-07-20T13:41:03.730498Z",
    "end_time": "2026-07-20T13:41:03.733631Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5,
        "num_results_found": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "b0aabf31-5ce9-4453-aebb-2b4e0d5a9256",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        

## Q4

In [9]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [10]:
provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))  # Using SQLite exporter, not console
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [14]:
rag_traced = RAGTraced(index=index, llm_client=llm_client)
result = rag_traced.rag("How does the agentic loop keep calling the model until it stops?")
answer, usage = result
print("Answer:", answer)

Answer: The loop keeps calling the model with `while True`, and after each response it checks whether the model returned any `function_call` items.

- If there are function calls, the code runs the tool, appends the tool output to `messages`, and loops again.
- If there are no function calls, it `break`s out of the loop.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: the model keeps deciding whether it needs another tool call, and your code keeps repeating until the model returns a final message with no tool calls.


In [15]:
conn = sqlite3.connect("traces.db")
cursor = conn.cursor()

# Get distinct span names
cursor.execute("SELECT DISTINCT name FROM spans")
span_names = cursor.fetchall()
print("Span names in database:")
for name in span_names:
    print(f"- {name[0]}")

# See count of each span name
cursor.execute("SELECT name, COUNT(*) as count FROM spans GROUP BY name")
span_counts = cursor.fetchall()
print("\nSpan counts:")
for name, count in span_counts:
    print(f"- {name}: {count}")

conn.close()

Span names in database:
- search
- llm
- rag

Span counts:
- llm: 4
- rag: 4
- search: 4


In [41]:
# Connect and check the table
conn = sqlite3.connect("traces.db")
cursor = conn.cursor()

# Check if the spans table exists
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(f"Tables in database: {tables}")


Tables in database: [('spans',)]


In [42]:
# Check the schema of spans table if it exists
if ('spans',) in tables:
    cursor.execute("PRAGMA table_info(spans);")
    schema = cursor.fetchall()
    print(f"Spans table schema: {schema}")

Spans table schema: [(0, 'name', 'TEXT', 0, None, 0), (1, 'start_time', 'INTEGER', 0, None, 0), (2, 'end_time', 'INTEGER', 0, None, 0), (3, 'input_tokens', 'INTEGER', 0, None, 0), (4, 'output_tokens', 'INTEGER', 0, None, 0), (5, 'cost', 'REAL', 0, None, 0)]


In [44]:
# Count rows
cursor.execute("SELECT COUNT(*) FROM spans;")
count = cursor.fetchone()[0]
print(f"Number of rows in spans: {count}")

Number of rows in spans: 0


## Q6

In [16]:
# Load the data
conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()

# Filter only llm spans
llm_spans = df[df['name'] == 'llm'].copy()
llm_spans.head()

,name,start_time,end_time,input_tokens,output_tokens,cost
1,llm,1784557750555545400,1784557752700870300,7121.0,127.0,None
4,llm,1784557754309228600,1784557755629207300,7121.0,93.0,None
7,llm,1784557757326056300,1784557759057852800,7121.0,124.0,None
10,llm,1784557763045232300,1784557764394906500,7121.0,127.0,None


In [17]:
llm_spans.describe()

,start_time,end_time,input_tokens,output_tokens
count,4.000000e+00,4.000000e+00,4.0,4.000000
mean,1.784558e+18,1.784558e+18,7121.0,117.750000
std,5.276127e+09,5.023396e+09,0.0,16.560495
min,1.784558e+18,1.784558e+18,7121.0,93.000000
25%,1.784558e+18,1.784558e+18,7121.0,116.250000
50%,1.784558e+18,1.784558e+18,7121.0,125.500000
75%,1.784558e+18,1.784558e+18,7121.0,127.000000
max,1.784558e+18,1.784558e+18,7121.0,127.000000


In [18]:
conn.close()